In [55]:
import numpy as np
import pandas as pd 

In [56]:
"""build_cooccurrence.py — full game x game co-occurrence similarity matrix.

sim(i,j) = co_count(i,j) / sqrt(users(i) * users(j))     (cosine over user sets)

Built from TRAIN only, to avoid leaking eval games into training features.
"""
import numpy as np
import pandas as pd
import scipy.sparse as sp


def build_cooccurrence_matrix(train_df, catalog_appids, save_path=None):
    """
    train_df:        DataFrame with columns ['steamid', 'appid']
    catalog_appids:  array of appids that define the matrix index order
    save_path:       optional .npz to persist the matrix

    Returns:
      sim (np.ndarray, float32)  — shape (n_games, n_games), sim[i,j] = similarity
      appid_to_idx (dict)        — appid -> row index in sim
    """
    appid_to_idx = {a: i for i, a in enumerate(catalog_appids)}
    n_games = len(catalog_appids)

    # ---- build the (users × games) binary matrix U
    users = train_df["steamid"].unique()
    user_to_idx = {u: i for i, u in enumerate(users)}

    u_idx = train_df["steamid"].map(user_to_idx).to_numpy(dtype = np.int64)
    g_idx = train_df["appid"].map(appid_to_idx).to_numpy(dtype = np.int64)

    valid = ~(pd.isna(u_idx) | pd.isna(g_idx))
    u_idx = u_idx[valid].astype(np.int64)
    g_idx = g_idx[valid].astype(np.int64)

    # dedupe (user, game) — a user may appear twice for the same game
    pairs = pd.DataFrame({"u": u_idx, "g": g_idx}).drop_duplicates()
    u_idx = pairs["u"].to_numpy()
    g_idx = pairs["g"].to_numpy()

    U = sp.csr_matrix(
        (np.ones(len(u_idx), dtype=np.float32), (u_idx, g_idx)),
        shape=(len(users), n_games),
    )
    print(f"  U: {U.shape}, nnz={U.nnz:,}")

    # ---- co-occurrence counts: C = Uᵀ U  (n_games × n_games)
    C = (U.T @ U).tocsr()
    print(f"  C: {C.shape}, nnz={C.nnz:,}")

    # ---- users(i) = number of users who have game i in their set
    users_per_game = np.asarray(U.sum(axis=0)).ravel().astype(np.float32)
    users_per_game[users_per_game == 0] = 1.0

    # ---- cosine normalization: sim = D^-1/2 · C · D^-1/2
    inv_sqrt = 1.0 / np.sqrt(users_per_game)
    C = C.tocsr().astype(np.float32)
    sim_sparse = sp.diags(inv_sqrt) @ C @ sp.diags(inv_sqrt)
    sim_sparse = sim_sparse.tocsr()

    # ---- densify to float32 (221 MB for 7430²)
    sim = sim_sparse.toarray().astype(np.float32)
    np.fill_diagonal(sim, 0.0)   # a game with itself is not a useful signal
    print(f"  sim: {sim.shape}, {sim.nbytes / 1e6:.0f} MB, "
          f"nonzero={np.count_nonzero(sim):,}")

    if save_path:
        np.savez_compressed(save_path, sim=sim,
                            appids=np.array(catalog_appids))
        print(f"  saved {save_path}")

    return sim, appid_to_idx

In [57]:
users_libraries = pd.read_csv("../data/training_data_train_final.csv")
games_catalog = pd.read_csv("../new_package/01_catalog/games_for_features.csv", index_col="appid")

In [58]:
print(users_libraries.columns)

Index(['steamid', 'appid', 'interest', 'slot1_n_labels', 'slot1_Action',
       'slot1_Adventure', 'slot1_Casual', 'slot1_Simulation', 'slot1_Strategy',
       'slot1_RPG',
       ...
       'cand_Emotional', 'cand_Retro', 'cand_Colorful', 'cand_Realistic',
       'cand_Singleplayer', 'cand_Multiplayer', 'cand_Co-op', 'cand_PvP',
       'cand_Local Multiplayer', 'cand_PvE'],
      dtype='object', length=673)


In [59]:
print(games_catalog.columns)

Index(['name', 'inclusion', 'popularity_rank', 'top10_users',
       'source_description', 'source_tags', 'short_description', 'description',
       'genres', 'tags', 'categories', 'positive', 'negative',
       'recommendations', 'pct_positive', 'peak_ccu', 'estimated_owners',
       'price', 'release_date', 'developers', 'publishers', 'has_description'],
      dtype='object')


In [60]:
print(users_libraries["appid"].dtype)

float64


In [61]:
users_libraries["appid"] = users_libraries["appid"].astype("int64")

In [62]:
catalog_appids = np.intersect1d(users_libraries["appid"].unique(), games_catalog.index.to_numpy())
appid_to_idx = {int(a): i for i, a in enumerate(catalog_appids)}

missing = users_libraries["appid"].map(appid_to_idx).isna().sum()
print(f"train rows whose appid isn't in appid_to_idx: {missing:,} / {len(users_libraries):,}")

train rows whose appid isn't in appid_to_idx: 0 / 117,489


In [63]:
train = users_libraries 
game_feats_df = games_catalog

In [64]:
catalog_appids = np.intersect1d(train["appid"].unique(), game_feats_df.index.to_numpy())
appid_to_idx = {int(a): i for i, a in enumerate(catalog_appids)}

# how many train rows map successfully?
mapped = train["appid"].map(lambda a: int(a) in appid_to_idx)
print(f"train rows mapping to catalog: {mapped.sum():,} / {len(train):,}")
print(f"unique train games: {train['appid'].nunique():,}")
print(f"catalog games: {len(catalog_appids):,}")
print(f"train games in catalog: {train['appid'].isin(catalog_appids).sum():,}")

# how many games per user?
games_per_user = train.groupby("steamid")["appid"].nunique()
print("\ngames per user:")
print(games_per_user.describe().round(2).to_string())

train rows mapping to catalog: 117,489 / 117,489
unique train games: 5,534
catalog games: 5,534
train games in catalog: 117,489

games per user:
count    20061.00
mean         5.86
std          0.76
min          1.00
25%          6.00
50%          6.00
75%          6.00
max         12.00


In [65]:
build_cooccurrence_matrix(users_libraries, games_catalog.index.values, save_path="../data/game_sim_matrix.npz")

  U: (20061, 7430), nnz=117,485
  C: (7430, 7430), nnz=219,754


  sim: (7430, 7430), 221 MB, nonzero=214,220
  saved ../data/game_sim_matrix.npz


(array([[0.        , 0.22642066, 0.22214545, ..., 0.        , 0.        ,
         0.        ],
        [0.22642063, 0.        , 0.31977358, ..., 0.        , 0.        ,
         0.        ],
        [0.22214545, 0.31977358, 0.        , ..., 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ]], shape=(7430, 7430), dtype=float32),
 {np.int64(730): 0,
  np.int64(440): 1,
  np.int64(4000): 2,
  np.int64(10): 3,
  np.int64(271590): 4,
  np.int64(105600): 5,
  np.int64(578080): 6,
  np.int64(550): 7,
  np.int64(240): 8,
  np.int64(218620): 9,
  np.int64(252490): 10,
  np.int64(227300): 11,
  np.int64(252950): 12,
  np.int64(1172470): 13,
  np.int64(570): 14,
  np.int64(304930): 15,
  np.int64(381210): 

In [66]:
"""eda_cooccurrence.py — inspect the co-occurrence similarity matrix."""
import numpy as np
import pandas as pd

# ---- load
data = np.load("../data/game_sim_matrix.npz")
sim = data["sim"]                       # (n_games, n_games) float32
appids = data["appids"]                 # (n_games,) int64
n_games = sim.shape[0]
print(f"shape: {sim.shape} | dtype: {sim.dtype} | memory: {sim.nbytes/1e6:.0f} MB")

# ---- names for readability
raw_games = pd.read_csv("../new_package/03_features/game_features.csv")
raw_games["appid"] = raw_games["appid"].astype("int64")
name_of = dict(zip(raw_games["appid"], raw_games["name"]))

# ---- global sparsity
nnz = np.count_nonzero(sim)
print(f"nonzero entries: {nnz:,} / {n_games**2:,} = {100*nnz/n_games**2:.4f}%")

# ---- per-game degree: how many nonzero neighbors does each game have?
degrees = (sim > 0).sum(axis=1)
print("\nneighbors per game (count of sim > 0):")
print(pd.Series(degrees).describe().round(2).to_string())

# ---- value distribution of nonzero similarities
vals = sim[sim > 0]
print("\nnonzero similarity values:")
print(pd.Series(vals).describe(percentiles=[.5,.9,.99,.999]).round(5).to_string())

shape: (7430, 7430) | dtype: float32 | memory: 221 MB
nonzero entries: 214,220 / 55,204,900 = 0.3880%

neighbors per game (count of sim > 0):
count    7430.00
mean       28.83
std       102.21
min         0.00
25%         0.00
50%         5.00
75%        17.00
max      3299.00

nonzero similarity values:
count    214220.00000
mean          0.04811
std           0.10092
min           0.00118
50%           0.01969
90%           0.09537
99%           0.57735
99.9%         1.00000
max           1.00000


In [67]:
# pick recognizable games (use Steam top sellers / the colleague's focus list)
FOCUS = [730, 440, 570, 105600, 252490, 271590, 359550, 1245620, 1086940, 413150]
idx_of = {a: i for i, a in enumerate(appids)}

def top_neighbors(appid, k=10):
    i = idx_of.get(appid)
    if i is None:
        return f"{appid} not in catalog"
    row = sim[i].copy()
    order = np.argsort(-row)[:k]
    out = []
    for j in order:
        if row[j] <= 0: break
        out.append((name_of.get(int(appids[j]), int(appids[j])),
                    float(row[j])))
    return out

for a in FOCUS:
    print(f"\n{a} — {name_of.get(a, '?')}")
    for name, s in top_neighbors(a, 10):
        print(f"   {s:.4f}  {name}")


730 — Counter-Strike 2
   0.3282  Counter-Strike
   0.2535  Grand Theft Auto V Legacy
   0.2460  PUBG: BATTLEGROUNDS
   0.2264  Team Fortress 2
   0.2229  Tom Clancy's Rainbow Six Siege
   0.2221  Garry's Mod
   0.1978  Counter-Strike: Source
   0.1819  Rust
   0.1803  Euro Truck Simulator 2
   0.1786  Left 4 Dead 2

440 — Team Fortress 2
   0.3198  Garry's Mod
   0.2264  Counter-Strike 2
   0.1996  Terraria
   0.1755  Left 4 Dead 2
   0.1487  PAYDAY 2
   0.1418  Fallout: New Vegas
   0.1297  Grand Theft Auto V Legacy
   0.1266  The Binding of Isaac: Rebirth
   0.1127  Deadlock
   0.1102  Tom Clancy's Rainbow Six Siege

570 — Dota 2
   0.1552  Counter-Strike 2
   0.1254  Counter-Strike
   0.0998  PUBG: BATTLEGROUNDS
   0.0832  Dota Underlords
   0.0722  Path of Exile
   0.0651  Grand Theft Auto V Legacy
   0.0602  Just Survive
   0.0601  Warframe
   0.0591  Team Fortress 2
   0.0579  Sid Meier's Civilization V

105600 — Terraria
   0.1996  Team Fortress 2
   0.1742  Garry's Mod
   0.1